# 01 — Databricks Ray + L4 Smoke Test

**Goal:** validate the full GPU path on this Databricks cluster *before* committing to the Ray retrofit of `src/cap/generator/`.

**Cluster expected:** SINGLE_USER, DBR 16.4 ML GPU, 1× L4 driver + 3× L4 workers (cluster `6106-192556-lj0uddmy`).

**What this verifies:**
1. Repo installs cleanly on the cluster (`pip install -e .`)
2. `ray.util.spark.setup_ray_cluster` brings up Ray across all 4 L4 GPUs
3. Flux.1-Dev + ControlNet + IP-Adapter loads on **one** L4 inside a Ray actor without OOM
4. One image generates end-to-end
5. (Optional) 4 actors generate 4 images in parallel — confirms the fan-out shape the full retrofit will use

**Expected wall time:** ~15–25 min on first run (most of it the ~24 GB Flux download). ~3–5 min on subsequent runs once cached.

**If this passes →** proceed with the Ray retrofit (Day 1 of the scoped plan). **If it OOMs or crashes →** we know exactly where, before we commit to a 30-hr full run.

## 1. Bootstrap

Install the repo in editable mode. Assumes this notebook is at `notebooks/01_databricks_ray_smoke.ipynb` inside the repo (e.g. cloned via Databricks Repos to `/Workspace/Repos/<user>/counterfactual-audit-pipeline`).

In [ ]:
%pip install -e .. --quiet

In [ ]:
dbutils.library.restartPython()

## 2. Cluster + GPU sanity check

In [ ]:
import sys
import torch
import ray

import cap
from cap.generator import FluxPuLIDControlNetGenerator, GenerationRequest

print(f"python:    {sys.version.split()[0]}")
print(f"cap:       {cap.__version__}")
print(f"torch:     {torch.__version__}")
print(f"cuda:      {torch.version.cuda}")
print(f"ray:       {ray.__version__}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU 0:     {p.name} | {p.total_memory / 1e9:.1f} GB VRAM")

## 3. HuggingFace cache + seed image

- `HF_HOME` → `/local_disk0/hf_cache` (per-node fast NVMe; survives within a cluster session, re-downloads on cluster restart). For production, swap to a Volume so all actors share a persistent cache.
- Download one stable public-domain test face for the smoke. Replace `SEED_IMAGE_URL` with your own face if you have one already on the cluster.

In [ ]:
import os
import urllib.request
from pathlib import Path

HF_CACHE = "/local_disk0/hf_cache"
SMOKE_DIR = Path("/local_disk0/cap_smoke")
SEED_IMAGE_PATH = SMOKE_DIR / "seed.jpg"
SEED_IMAGE_URL = "https://raw.githubusercontent.com/InstantID/InstantID/main/examples/yann-lecun_resize.jpg"

os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
Path(HF_CACHE).mkdir(parents=True, exist_ok=True)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

if not SEED_IMAGE_PATH.exists():
    urllib.request.urlretrieve(SEED_IMAGE_URL, SEED_IMAGE_PATH)

print(f"HF cache:   {HF_CACHE}")
print(f"Seed image: {SEED_IMAGE_PATH} ({SEED_IMAGE_PATH.stat().st_size / 1024:.0f} KB)")

SEED_IMAGE_BYTES = SEED_IMAGE_PATH.read_bytes()

## 4. Start Ray on the Spark cluster

`setup_ray_cluster` lays Ray on top of the existing Spark workers. With `num_gpus_head_node=1` the driver L4 also joins the actor pool → 4 actors total. If the driver-GPU path is finicky on your DBR build, drop `num_gpus_head_node=0` and accept 3 actors.

In [ ]:
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(
    num_worker_nodes=3,
    num_gpus_worker_node=1,
    num_cpus_worker_node=8,
    num_gpus_head_node=1,
    num_cpus_head_node=4,
)

ray.init(ignore_reinit_error=True)
print("Ray cluster resources:")
for k, v in sorted(ray.cluster_resources().items()):
    print(f"  {k}: {v}")

## 5. Define the FluxActor (wraps the existing generator)

Inline definition for smoke. The production version of this lands in `src/cap/generator/ray_runner.py` during the Day-1 retrofit — the shape is identical.

In [ ]:
@ray.remote(num_gpus=1)
class FluxActor:
    def __init__(self, dtype="fp8", controlnet_mode="pose", cache_dir="/local_disk0/hf_cache"):
        import os
        os.environ["HF_HOME"] = cache_dir
        os.environ["HUGGINGFACE_HUB_CACHE"] = cache_dir
        os.environ["TRANSFORMERS_CACHE"] = cache_dir

        from cap.generator import FluxPuLIDControlNetGenerator
        self.generator = FluxPuLIDControlNetGenerator(
            dtype=dtype,
            controlnet_mode=controlnet_mode,
            cache_dir=cache_dir,
        )

    def warm(self):
        import time, torch
        t0 = time.time()
        self.generator._lazy_load()
        load_s = time.time() - t0
        vram_mb = torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0
        return {
            "load_s": round(load_s, 1),
            "vram_mb_after_load": round(vram_mb, 0),
            "model_versions": self.generator.model_versions(),
        }

    def generate_one(self, request_kwargs, seed_image_bytes, output_dir="/local_disk0/cap_smoke_out"):
        from pathlib import Path
        from cap.generator import GenerationRequest

        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        if seed_image_bytes is not None:
            seed_path = out_dir / "_seed.jpg"
            seed_path.write_bytes(seed_image_bytes)
            request_kwargs = {**request_kwargs, "seed_image_path": str(seed_path)}

        req = GenerationRequest(**request_kwargs)
        results = self.generator.generate(req, out_dir)
        out = []
        for r in results:
            png_bytes = Path(r.image_path).read_bytes()
            out.append({
                "counterfactual_id": r.counterfactual_id,
                "prompt_used": r.prompt_used,
                "axis_values": r.axis_values,
                "elapsed_s": r.metadata.get("elapsed_s"),
                "png_bytes": png_bytes,
            })
        return out

## 6. Warm one actor (loads Flux into VRAM)

First call downloads ~24 GB of weights into the HF cache (slow, one-time). Subsequent actors on the same node will hit the cache. Watch `vram_mb_after_load` — should land around 16–20 GB on FP8.

In [ ]:
import time

actor = FluxActor.remote(dtype="fp8", controlnet_mode="pose")

t0 = time.time()
warm_info = ray.get(actor.warm.remote())
print(f"Warm wall time: {time.time() - t0:.1f}s")
print(f"Model load:     {warm_info['load_s']}s")
print(f"VRAM after load: {warm_info['vram_mb_after_load']:.0f} MB")
print(f"Model versions:")
for k, v in warm_info["model_versions"].items():
    print(f"  {k}: {v}")

## 7. Generate one image

Tiny config: 768×768, 20 inference steps, 1 demographic axis with 1 value → exactly one image. Should take ~20–40s on a warm L4.

In [ ]:
request_kwargs = dict(
    seed_identity_id="smoke_001",
    seed_image_path=None,  # filled in by the actor from seed_image_bytes
    seed_prompt=None,
    counterfactual_axes={"skin_tone": [3]},
    fixed_attributes={
        "pose": "frontal",
        "expression": "neutral expression",
        "lighting": "soft even studio lighting",
    },
    seed=42,
    num_inference_steps=20,
    guidance_scale=3.5,
    width=768,
    height=768,
)

t0 = time.time()
results = ray.get(actor.generate_one.remote(request_kwargs, SEED_IMAGE_BYTES))
print(f"Generation wall time: {time.time() - t0:.1f}s")
for r in results:
    print(f"  {r['counterfactual_id']} | {r['elapsed_s']}s | {len(r['png_bytes']) / 1024:.0f} KB")
    print(f"  prompt: {r['prompt_used']}")

## 8. Display the result

In [ ]:
import io
from PIL import Image

img = Image.open(io.BytesIO(results[0]["png_bytes"]))
display(img)

## 9. (Optional) 4-actor fan-out

Validates the full-retrofit pattern: 4 actors, each holding Flux in VRAM, processing requests in parallel via `ActorPool`. Costs ~4× the GPU time of the single-actor smoke (each actor downloads/loads its own copy of the weights), so skip on first run if you just want the basic path verified.

Uncomment to run.

In [ ]:
# from ray.util import ActorPool
#
# # Reuse the warm actor + spin up 3 more
# extra_actors = [FluxActor.remote(dtype="fp8", controlnet_mode="pose") for _ in range(3)]
# ray.get([a.warm.remote() for a in extra_actors])  # parallel warm
# pool = ActorPool([actor, *extra_actors])
#
# # 4 distinct requests (varying skin_tone)
# requests = []
# for st in [1, 3, 4, 6]:
#     rk = {**request_kwargs, "counterfactual_axes": {"skin_tone": [st]}, "seed_identity_id": f"smoke_st{st}"}
#     requests.append((rk, SEED_IMAGE_BYTES))
#
# t0 = time.time()
# results_4 = list(pool.map_unordered(
#     lambda actor, payload: actor.generate_one.remote(*payload),
#     requests,
# ))
# wall = time.time() - t0
# flat = [r for batch in results_4 for r in batch]
# print(f"4-image fan-out wall: {wall:.1f}s")
# print(f"Mean per-image elapsed (actor-side): {sum(r['elapsed_s'] for r in flat) / len(flat):.1f}s")
# print(f"Speedup vs serial:    {sum(r['elapsed_s'] for r in flat) / wall:.2f}x")

## 10. Cleanup

In [ ]:
shutdown_ray_cluster()
print("Ray cluster shut down.")

## What success looks like

- ✓ `setup_ray_cluster` printed 4 GPUs in `ray.cluster_resources()`
- ✓ Actor warmed without OOM, VRAM landed in the 16–22 GB range
- ✓ One image generated and rendered above
- ✓ (If section 9 ran) 4 actors generated in parallel, speedup ≥ ~3×

## If it OOMed

- Drop `dtype="fp8"` → `dtype="nf4"` in the actor (4-bit weights, smaller)
- Drop resolution to 512×512 in `request_kwargs`
- Reduce `num_inference_steps`
- If still OOM → the L4 24 GB envelope is too tight for Flux + ControlNet + IP-Adapter together. Fallback: drop ControlNet for the smoke and re-test with raw Flux.

## Next steps

1. Move the inline `FluxActor` into `src/cap/generator/ray_runner.py`
2. Add `run_distributed(requests, output_dir, num_actors=4)` that builds an `ActorPool` and writes a per-image manifest as results stream in
3. Add `--backend ray` to `src/cap/cli/generate.py`
4. Move HF cache to a Workspace Volume so all actors share the download (avoids 4× cold loads)
5. Add idempotent skip-if-exists on output paths so cluster restarts are free